In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!unzip -oq /content/drive/MyDrive/leukemia_project/TXL-PBC.zip -d /content/

In [ ]:
from pathlib import Path

root = Path("/content/TXL-PBC")

print("Found:", root.exists())
print("Train:", len(list((root / "images/train").glob("*.png"))))
print("Validation:", len(list((root / "images/val").glob("*.png"))))

In [ ]:
Path("/content/txl_pbc_local.yaml").write_text("""
path: /content/TXL-PBC
train: images/train
val: images/val

names:
  0: WBC
  1: RBC
  2: Platelet
""")

**Because day1 I trained a YOLO model for txl-pbc so, Today I will use best.pt to predict images**

In [ ]:
from pathlib import Path
import torch

checkpoint = Path(
    "/content/runs/txl_pbc_t4_local/weights/best.pt"
)
data_root = Path("/content/TXL-PBC")

print("GPU Ready:", torch.cuda.is_available())
print("Has checkpoint:", checkpoint.is_file())
print("Has data:", data_root.is_dir())

**Luckily, I saved a copy in my drive**

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

run_dir = Path(
    "/content/drive/MyDrive/leukemia_project/txl_pbc_t4_local"
)

candidates = sorted(run_dir.rglob("best.pt"))

if not candidates:
    raise FileNotFoundError(f"No best.pt found in {run_dir}")

for path in candidates:
    print("Checkpoint found:", path)

if len(candidates) != 1:
    raise RuntimeError("Multiple checkpoints found. Select the intended path.")

checkpoint = candidates[0]
print("Selected checkpoint:", checkpoint)

In [ ]:
import torch

data_root = Path("/content/TXL-PBC")

print("CUDA available:", torch.cuda.is_available())
print("Dataset available:", data_root.is_dir())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

for split in ("train", "val"):
    image_dir = data_root / "images" / split
    print(f"{split} PNG images:", len(list(image_dir.glob("*.png"))))

In [ ]:
!pip install ultralytics

**Test predict 3 images**

In [ ]:
from ultralytics import YOLO
from IPython.display import display
from PIL import Image

model = YOLO(str(checkpoint))

image_paths = sorted(
    (data_root / "images" / "val").glob("*.png")
)[:3]

assert image_paths, "Not found image validation"

for image_path in image_paths:
    result = model.predict(
        source=str(image_path),
        imgsz=640,
        conf=0.25,
        device=0 if torch.cuda.is_available() else "cpu",
        verbose=False,
    )[0]

    print(image_path.name)
  
    display(Image.fromarray(result.plot()[..., ::-1]))

**Crop 3 WBC**

In [ ]:
import math
from pathlib import Path

import torch
from PIL import Image
from IPython.display import display

output_dir = Path("/content/wbc_crops_preview")
output_dir.mkdir(parents=True, exist_ok=True)

for image_path in image_paths:
    result = model.predict(
        source=str(image_path),
        imgsz=640,
        conf=0.25,
        classes=[0],  # Class 0 is WBC in this model
        device=0 if torch.cuda.is_available() else "cpu",
        verbose=False,
    )[0]

    with Image.open(image_path) as source:
        original = source.convert("RGB")

    width, height = original.size

    for index, box in enumerate(result.boxes):
        left, top, right, bottom = box.xyxy[0].cpu().tolist()

        # Round outward and keep coordinates inside the image.
        x1 = max(0, math.floor(left))
        y1 = max(0, math.floor(top))
        x2 = min(width, math.ceil(right))
        y2 = min(height, math.ceil(bottom))

        if x2 <= x1 or y2 <= y1:
            continue

        crop = original.crop((x1, y1, x2, y2))
        output_path = output_dir / f"{image_path.stem}_wbc_{index}.png"
        crop.save(output_path)

        confidence = box.conf[0].item()
        print(f"Saved: {output_path.name}")
        print(f"Size: {crop.size}, confidence: {confidence:.3f}")
        display(crop)

**Resize Image**

In [ ]:
from PIL import Image, ImageOps
from IPython.display import display

def prepare_square(image, size=224):
    resized = ImageOps.contain(
        image,
        (size, size),
        method=Image.Resampling.LANCZOS,
    )

    canvas = Image.new("RGB", (size, size), color=(0, 0, 0))
    left = (size - resized.width) // 2
    top = (size - resized.height) // 2
    canvas.paste(resized, (left, top))

    return canvas


for crop_path in sorted(output_dir.glob("*.png")):
    with Image.open(crop_path) as source:
        crop = source.convert("RGB")

    prepared = prepare_square(crop)

    print(f"{crop_path.name}: {crop.size} -> {prepared.size}")
    display(prepared)

**Create batch from 3 crop images**

In [ ]:
import numpy as np
import torch
from PIL import Image

image_tensors = []

for crop_path in sorted(output_dir.glob("*.png")):
    with Image.open(crop_path) as source:
        image = source.convert("RGB")

    prepared = prepare_square(image, size=224)

    # Convert RGB pixels to float32 values in [0, 1].
    array = np.asarray(prepared, dtype=np.float32) / 255.0  #scale the value of each color channel from the range [0, 255] to [0, 1]

    # Change layout from HWC to CHW.
    tensor = torch.from_numpy(array).permute(2, 0, 1).contiguous()
    image_tensors.append(tensor)

if not image_tensors:
    raise RuntimeError("No crop images found.")

batch = torch.stack(image_tensors)

print("Batch shape:", batch.shape)
print("Data type:", batch.dtype)
print("Minimum value:", batch.min().item())
print("Maximum value:", batch.max().item())